In [56]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [57]:
# --- 1 ---
df = pd.read_csv('wells_info_with_prod.csv')

date_cols = ['PermitDate', 'SpudDate', 'CompletionDate', 'FirstProductionDate']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

In [58]:
# Целевая переменная
Y = df['Prod1Year']
df = df.drop(['Prod1Year'], axis=1)

In [59]:
# Оставляем CompletionDate и formation:
date_keep = ['CompletionDate']
cat_keep = ['formation']

# Остальные категориальные/датовые будем кодировать one-hot
cat_ohe_cols = ['FirstProductionDate', 'operatorNameIHS', 'BasinName', 'StateName', 'CountyName']

# Удалим даты, которые не используются
drop_dates = ['PermitDate', 'SpudDate']
df = df.drop(columns=[c for c in drop_dates if c in df.columns])


In [60]:
# Берём данные для one-hot кодирования
ohe_input_cols = [c for c in cat_ohe_cols if c in df.columns]
cat_data_for_ohe = df[ohe_input_cols]

# Уберём их из df, чтобы там остались:
# числовые признаки, CompletionDate, formation
df = df.drop(columns=ohe_input_cols)

In [61]:
# One-hot кодирование
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
one_hot_array = ohe.fit_transform(cat_data_for_ohe)
one_hot_cols = ohe.get_feature_names_out(ohe_input_cols)
one_hot_data = pd.DataFrame(one_hot_array, index=cat_data_for_ohe.index, columns=one_hot_cols)


# Итоговая матрица признаков X:
# числовые + CompletionDate + formation + one-hot
X = pd.concat([df, one_hot_data], axis=1)

# print(X[['CompletionDate', 'formation']].head())



In [71]:
X

,API,CompletionDate,formation,LatWGS84,LonWGS84,BottomHoleLatitude,BottomHoleLongitude,LATERAL_LENGTH_BLEND,PROP_PER_FOOT,WATER_PER_FOOT,...,CountyName_LOVING,CountyName_MARTIN,CountyName_MCKENZIE,CountyName_MOUNTRAIL,CountyName_PECOS,CountyName_REAGAN,CountyName_REEVES,CountyName_UPTON,CountyName_WELD,CountyName_WILLIAMS
0,5005072170100,2014-12-02,NIOBRARA,39.684606,-104.642128,39.68445,-104.60557,9005.0,994.6866,591.800400,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5123377130000,2014-02-26,NIOBRARA,40.509320,-104.780980,40.49692,-104.77859,4195.0,991.5857,628.632100,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,5123379280000,2014-09-07,NIOBRARA,40.335390,-104.363000,40.34780,-104.36863,4273.0,1000.2760,564.484100,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,5123379400000,2015-03-31,NIOBRARA,40.152220,-104.530780,40.17445,-104.52932,7078.0,973.4437,824.002000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,5123385820100,2014-04-23,NIOBRARA,40.508303,-104.868180,40.49558,-104.86757,3211.0,783.5919,603.141400,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5,5123390320000,2014-10-30,NIOBRARA,40.520989,-104.450861,40.52100,-104.46772,4259.0,1011.0440,698.681900,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,5123390440000,2014-08-16,NIOBRARA,40.118872,-104.789372,40.13079,-104.78716,4518.0,469.8262,798.167600,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,5123392690000,2015-08-03,NIOBRARA,40.147015,-104.898342,40.16105,-104.89340,4975.0,1267.8870,835.992800,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8,5123399750000,2015-04-16,NIOBRARA,40.263373,-104.727955,40.27426,-104.73307,4786.0,805.9642,1133.333000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
9,5123402600000,2015-09-13,CODELL,40.353967,-104.944679,40.35440,-104.96173,4195.0,926.5010,403.322300,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [62]:
# --- 2 ---

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [ ]:
# --- 3 --- 

scaler_X = StandardScaler()

# Выбираем числовые столбцы (int/float, включая one-hot)
num_cols = X.select_dtypes(include=[np.number]).columns

# Отдельно берём числовую часть train/test
X_train_num = X_train[num_cols]
X_test_num = X_test[num_cols]

# Масштабируем только числовые признаки
X_train_num_scaled = scaler_X.fit_transform(X_train_num)
X_test_num_scaled = scaler_X.transform(X_test_num)


# Собираем обратно DataFrame, оставляя немасштабированные дату и категориальный признак
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = X_train_num_scaled
X_test_scaled[num_cols] = X_test_num_scaled

# Теперь в X_train_scaled / X_test_scaled:
# CompletionDate и formation остались как есть
# остальные числовые (включая one-hot) отмасштабированы


In [66]:
# --- 4 ---
scaler_Y = StandardScaler()

Y_train_2d = Y_train.values.reshape(-1, 1)
Y_test_2d = Y_test.values.reshape(-1, 1)

Y_train_scaled = scaler_Y.fit_transform(Y_train_2d)
Y_test_scaled = scaler_Y.transform(Y_test_2d)

# Превращаем обратно в вектор (1D)
Y_train_scaled = Y_train_scaled.ravel()
Y_test_scaled = Y_test_scaled.ravel()

# print(X_train_scaled.shape, X_test_scaled.shape)
# print(Y_train_scaled.shape, Y_test_scaled.shape)